
# **Quarto aplicado a dados de saúde**



Este notebook foi preparado para uma aula prática no Google Colab.


## **Objetivos**


Ao final, o participante deverá ser capaz de:

1. Entender a diferença entre um notebook de análise e um relatório publicado.
2. Gerar uma base fictícia no contexto da atenção primária.
3. Fazer uma análise exploratória e uma modelagem simples.
4. Criar um documento `.qmd`.
5. Renderizar o documento em HTML com o Quarto.
6. Identificar recursos úteis do Quarto, como sumário, seções numeradas, código recolhível, abas, alertas, tabelas e referências cruzadas.

> Todos os dados deste exercício são inteiramente fictícios e não representam pessoas reais.

Documentação Quarto: https://quarto.org/docs/reference/


## **1. Instalação das ferramentas**



O Colab já oferece Python e Jupyter. Precisamos instalar o Quarto e as bibliotecas usadas na análise.

A instalação abaixo utiliza o pacote oficial mais recente para Linux. O comando `quarto check` confirma se a instalação funcionou.

In [1]:

# Instala o Quarto no ambiente temporário do Google Colab
!wget -q https://quarto.org/download/latest/quarto-linux-amd64.deb -O quarto.deb
!dpkg -i quarto.deb > /dev/null
!quarto check

# Bibliotecas da análise
!pip -q install pandas numpy matplotlib plotly statsmodels scikit-learn tabulate


Quarto 1.10.18
[✓] Checking environment information...
      Quarto cache location: /root/.cache/quarto
[✓] Checking versions of quarto binary dependencies...
      Pandoc version 3.10.0: OK
      Dart Sass version 1.101.0: OK
      Deno version 2.7.14: OK
      Typst version 0.15.1: OK
[✓] Checking versions of quarto dependencies......OK
[✓] Checking Quarto installation......OK
      Version: 1.10.18
      Path: /opt/quarto/bin

[✓] Checking tools....................OK
      TinyTeX: (not installed)
      Chrome Headless Shell: (not installed)
      VeraPDF: (not installed)

[✓] Checking LaTeX....................OK
      Tex:  (not detected)

[✓] Checking Chrome Headless....................OK
      Chrome:  (not detected)

[✓] Checking basic markdown render....OK

[✓] Checking R installation...........OK
      Version: 4.6.1
      Path: /usr/lib/R
      LibPaths:
        - /usr/local/lib/R/site-library
        - /usr/lib/R/site-library
        - /usr/lib/R/library
      knitr: 1.51
  


## **2. Bibliotecas e configurações**



Usamos uma semente fixa para que a base fictícia seja reproduzível. Assim, todos que executarem o notebook obterão os mesmos dados.

In [2]:

import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import statsmodels.api as sm

from IPython.display import HTML, display

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

PASTA_PROJETO = Path("/content/quarto_saude")
PASTA_PROJETO.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

print(f"Pasta do projeto: {PASTA_PROJETO}")


Pasta do projeto: /content/quarto_saude



## **3. Geração da base fictícia**


A base representa pacientes adultos vinculados a unidades de atenção primária.

Variáveis principais

- `idade`
- `sexo`
- `regiao`
- `indice_vulnerabilidade`
- `diabetes`
- `hipertensao`
- `consultas_aps_12m`
- `consultas_urgencia_12m`
- `acompanhamento_regular`
- `internacao_evitavel_12m`
- `custo_total_12m`

As relações entre as variáveis são simuladas propositalmente. Por exemplo, maior vulnerabilidade, doenças crônicas e menor acompanhamento aumentam a probabilidade de internação evitável.


In [3]:

N = 3000

idade = np.clip(rng.normal(52, 18, N).round(), 18, 95).astype(int)
sexo = rng.choice(["Feminino", "Masculino"], size=N, p=[0.55, 0.45])
regiao = rng.choice(
    ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"],
    size=N,
    p=[0.09, 0.28, 0.09, 0.42, 0.12],
)

# Índice entre 0 e 1. Valores maiores indicam maior vulnerabilidade.
efeito_regiao_vulnerabilidade = {
    "Norte": 0.15,
    "Nordeste": 0.12,
    "Centro-Oeste": 0.02,
    "Sudeste": -0.05,
    "Sul": -0.07,
}
indice_vulnerabilidade = np.clip(
    rng.beta(2.2, 3.0, N)
    + np.array([efeito_regiao_vulnerabilidade[r] for r in regiao]),
    0,
    1,
)

prob_hipertensao = 1 / (
    1 + np.exp(-(-5.0 + 0.065 * idade + 1.0 * indice_vulnerabilidade))
)
hipertensao = rng.binomial(1, prob_hipertensao)

prob_diabetes = 1 / (
    1 + np.exp(-(-5.4 + 0.052 * idade + 0.9 * indice_vulnerabilidade + 0.55 * hipertensao))
)
diabetes = rng.binomial(1, prob_diabetes)

prob_acompanhamento = 1 / (
    1 + np.exp(-(
        0.9
        - 1.8 * indice_vulnerabilidade
        + 0.30 * hipertensao
        + 0.35 * diabetes
    ))
)
acompanhamento_regular = rng.binomial(1, prob_acompanhamento)

media_consultas_aps = (
    1.3
    + 1.7 * acompanhamento_regular
    + 1.0 * hipertensao
    + 1.2 * diabetes
    + 0.012 * idade
)
consultas_aps_12m = rng.poisson(media_consultas_aps)

media_urgencia = np.exp(
    -0.7
    + 1.0 * indice_vulnerabilidade
    + 0.35 * hipertensao
    + 0.50 * diabetes
    - 0.35 * acompanhamento_regular
)
consultas_urgencia_12m = rng.poisson(media_urgencia)

logit_internacao = (
    -4.4
    + 1.9 * indice_vulnerabilidade
    + 0.65 * hipertensao
    + 0.90 * diabetes
    + 0.32 * consultas_urgencia_12m
    - 0.55 * acompanhamento_regular
    + 0.012 * (idade - 50)
)
prob_internacao = 1 / (1 + np.exp(-logit_internacao))
internacao_evitavel_12m = rng.binomial(1, prob_internacao)

custo_total_12m = (
    350
    + 95 * consultas_aps_12m
    + 310 * consultas_urgencia_12m
    + 7200 * internacao_evitavel_12m
    + 420 * diabetes
    + 260 * hipertensao
    + rng.gamma(2, 180, N)
).round(2)

df = pd.DataFrame(
    {
        "id_paciente": [f"P{i:05d}" for i in range(1, N + 1)],
        "idade": idade,
        "sexo": sexo,
        "regiao": regiao,
        "indice_vulnerabilidade": indice_vulnerabilidade.round(3),
        "hipertensao": hipertensao,
        "diabetes": diabetes,
        "acompanhamento_regular": acompanhamento_regular,
        "consultas_aps_12m": consultas_aps_12m,
        "consultas_urgencia_12m": consultas_urgencia_12m,
        "internacao_evitavel_12m": internacao_evitavel_12m,
        "custo_total_12m": custo_total_12m,
    }
)

# Cria pequena quantidade de ausências para demonstrar verificação de qualidade.
for coluna, percentual in {
    "indice_vulnerabilidade": 0.025,
    "acompanhamento_regular": 0.015,
}.items():
    indices = rng.choice(df.index, size=int(N * percentual), replace=False)
    df.loc[indices, coluna] = np.nan

arquivo_csv = PASTA_PROJETO / "pacientes_ficticios.csv"
df.to_csv(arquivo_csv, index=False)

print(f"Base salva em: {arquivo_csv}")
print(f"Dimensão: {df.shape[0]:,} linhas e {df.shape[1]} colunas")
df.head()


Base salva em: /content/quarto_saude/pacientes_ficticios.csv
Dimensão: 3,000 linhas e 12 colunas


,id_paciente,idade,sexo,regiao,indice_vulnerabilidade,hipertensao,diabetes,acompanhamento_regular,consultas_aps_12m,consultas_urgencia_12m,internacao_evitavel_12m,custo_total_12m
0,P00001,57,Masculino,Norte,0.832,0,1,1.0,3,1,0,2441.36
1,P00002,33,Masculino,Norte,0.444,0,0,1.0,1,0,0,478.94
2,P00003,66,Masculino,Nordeste,0.296,0,0,1.0,6,1,0,1440.80
3,P00004,69,Masculino,Sudeste,0.625,1,0,0.0,2,2,0,2251.62
4,P00005,18,Masculino,Sudeste,0.215,0,0,1.0,2,2,0,1515.88



## **4. Análise preliminar no próprio notebook**



Antes de publicar, verificamos se os dados e os resultados fazem sentido.

In [4]:

resumo_qualidade = pd.DataFrame(
    {
        "tipo": df.dtypes.astype(str),
        "ausentes_n": df.isna().sum(),
        "ausentes_pct": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(dropna=True),
    }
)

resumo_qualidade


,tipo,ausentes_n,ausentes_pct,valores_unicos
id_paciente,object,0,0.0,3000
idade,int64,0,0.0,78
sexo,object,0,0.0,2
regiao,object,0,0.0,5
indice_vulnerabilidade,float64,75,2.5,836
hipertensao,int64,0,0.0,2
diabetes,int64,0,0.0,2
acompanhamento_regular,float64,45,1.5,2
consultas_aps_12m,int64,0,0.0,14
consultas_urgencia_12m,int64,0,0.0,9


In [5]:

indicadores = pd.Series(
    {
        "Pacientes": len(df),
        "Idade média": df["idade"].mean(),
        "Prevalência de hipertensão (%)": 100 * df["hipertensao"].mean(),
        "Prevalência de diabetes (%)": 100 * df["diabetes"].mean(),
        "Acompanhamento regular (%)": 100 * df["acompanhamento_regular"].mean(),
        "Internação evitável (%)": 100 * df["internacao_evitavel_12m"].mean(),
        "Custo médio anual (R$)": df["custo_total_12m"].mean(),
    }
).round(2)

indicadores.to_frame("valor")


,valor
Pacientes,3000.00
Idade média,51.73
Prevalência de hipertensão (%),27.17
Prevalência de diabetes (%),13.90
Acompanhamento regular (%),54.82
Internação evitável (%),5.40
Custo médio anual (R$),1789.95


In [6]:

analise_regiao = (
    df.groupby("regiao", observed=True)
    .agg(
        pacientes=("id_paciente", "count"),
        idade_media=("idade", "mean"),
        vulnerabilidade_media=("indice_vulnerabilidade", "mean"),
        acompanhamento_pct=("acompanhamento_regular", lambda x: 100 * x.mean()),
        internacao_pct=("internacao_evitavel_12m", lambda x: 100 * x.mean()),
        custo_medio=("custo_total_12m", "mean"),
    )
    .round(2)
    .sort_values("internacao_pct", ascending=False)
)

analise_regiao


,pacientes,idade_media,vulnerabilidade_media,acompanhamento_pct,internacao_pct,custo_medio
regiao,,,,,,
Norte,260,53.35,0.56,47.66,8.08,2033.48
Nordeste,828,52.30,0.54,46.26,6.28,1885.01
Sudeste,1247,51.26,0.37,58.98,5.05,1730.88
Centro-Oeste,293,52.25,0.46,61.54,4.10,1716.98
Sul,372,50.46,0.36,59.67,3.76,1663.59


In [7]:

fig = px.bar(
    analise_regiao.reset_index(),
    x="regiao",
    y="internacao_pct",
    text_auto=".1f",
    title="Internações evitáveis por região fictícia",
    labels={
        "regiao": "Região",
        "internacao_pct": "Pacientes com internação evitável (%)",
    },
)
fig.update_layout(yaxis_rangemode="tozero")
fig.show()



## **5. Modelo explicativo simples**




O objetivo não é produzir inferência causal. Usamos regressão logística apenas para ilustrar como o Quarto reúne metodologia, código, resultado e interpretação no mesmo documento.

In [8]:

df_modelo = (
    df[
        [
            "internacao_evitavel_12m",
            "idade",
            "indice_vulnerabilidade",
            "hipertensao",
            "diabetes",
            "acompanhamento_regular",
            "consultas_urgencia_12m",
        ]
    ]
    .dropna()
    .copy()
)

X = df_modelo.drop(columns="internacao_evitavel_12m")
X = sm.add_constant(X)
y = df_modelo["internacao_evitavel_12m"]

modelo = sm.Logit(y, X).fit(disp=False)

resultado_modelo = pd.DataFrame(
    {
        "variavel": modelo.params.index,
        "odds_ratio": np.exp(modelo.params.values),
        "ic95_inferior": np.exp(modelo.conf_int()[0].values),
        "ic95_superior": np.exp(modelo.conf_int()[1].values),
        "p_valor": modelo.pvalues.values,
    }
).round(3)

resultado_modelo


,variavel,odds_ratio,ic95_inferior,ic95_superior,p_valor
0,const,0.012,0.006,0.027,0.000
1,idade,1.010,0.999,1.022,0.072
2,indice_vulnerabilidade,2.847,1.251,6.479,0.013
3,hipertensao,1.897,1.288,2.795,0.001
4,diabetes,2.767,1.872,4.090,0.000
5,acompanhamento_regular,0.454,0.317,0.651,0.000
6,consultas_urgencia_12m,1.372,1.189,1.584,0.000



## **6. Publicação multiplataforma com Quarto**




HTML, PDF, Word e PowerPoint não possuem o mesmo comportamento.

- O HTML aceita gráficos interativos em JavaScript.
- PDF, Word e PowerPoint precisam de gráficos estáticos, como PNG, SVG ou PDF vetorial.
- Um relatório longo não deve ser convertido diretamente em apresentação. Slides precisam de menos texto, tabelas menores e uma seção principal por tela.

Por isso, criaremos dois arquivos:

1. `relatorio_saude.qmd`, para HTML, Word e PDF.
2. `apresentacao_saude.qmd`, para PowerPoint.

Os dois usam a mesma base e os mesmos cálculos.

In [9]:

# Cria o relatório e a apresentação como documentos separados.
arquivo_relatorio = PASTA_PROJETO / "relatorio_saude.qmd"
arquivo_apresentacao = PASTA_PROJETO / "apresentacao_saude.qmd"

arquivo_relatorio.write_text('\n---\ntitle: "Relatório de Indicadores da Atenção Primária"\nsubtitle: "Demonstração do Quarto com dados inteiramente fictícios"\nauthor: "Material didático"\ndate: today\nlang: pt-BR\nformat:\n  html:\n    theme: cosmo\n    toc: true\n    toc-location: left\n    toc-depth: 3\n    number-sections: true\n    code-fold: true\n    code-summary: "Mostrar o código"\n    code-tools: true\n    embed-resources: true\n  docx:\n    toc: true\n    number-sections: true\n  pdf:\n    toc: true\n    number-sections: true\n    documentclass: article\n    geometry:\n      - top=2cm\n      - bottom=2cm\n      - left=2cm\n      - right=2cm\n    keep-tex: false\nexecute:\n  echo: false\n  warning: false\n  error: false\njupyter: python3\n---\n\n# Apresentação\n\nEste relatório demonstra como uma análise de dados de saúde pode ser convertida em um produto publicável e reproduzível.\n\n::: {.callout-important}\n## Dados fictícios\n\nA base foi criada exclusivamente para fins educacionais. Nenhuma linha representa um paciente real.\n:::\n\n# Preparação dos dados\n\n```{python}\n#| label: setup\n#| include: false\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport statsmodels.api as sm\nfrom IPython.display import Markdown, display\n\nCAMINHO_BASE = Path("pacientes_ficticios.csv")\ndf = pd.read_csv(CAMINHO_BASE)\n\nplt.rcParams.update({\n    "figure.figsize": (8, 4.8),\n    "figure.dpi": 130,\n    "axes.spines.top": False,\n    "axes.spines.right": False,\n})\n```\n\nA base contém `{python} f"{len(df):,}".replace(",", ".")` pacientes e `{python} df.shape[1]` variáveis.\n\n## Qualidade dos dados\n\n```{python}\n#| label: tbl-qualidade\n#| tbl-cap: "Completude das variáveis da base fictícia"\n\nnomes_variaveis = {\n    "id_paciente": "Identificador",\n    "idade": "Idade",\n    "sexo": "Sexo",\n    "regiao": "Região",\n    "indice_vulnerabilidade": "Índice de vulnerabilidade",\n    "hipertensao": "Hipertensão",\n    "diabetes": "Diabetes",\n    "acompanhamento_regular": "Acompanhamento regular",\n    "consultas_aps_12m": "Consultas na APS",\n    "consultas_urgencia_12m": "Consultas de urgência",\n    "internacao_evitavel_12m": "Internação evitável",\n    "custo_total_12m": "Custo total",\n}\n\nqualidade = pd.DataFrame({\n    "Variável": [nomes_variaveis[c] for c in df.columns],\n    "Ausentes": df.isna().sum().values,\n    "Ausentes (%)": (100 * df.isna().mean()).round(1).values,\n})\n\ndisplay(qualidade)\n```\n\nA @tbl-qualidade mostra que as ausências foram pequenas e introduzidas propositalmente.\n\n# Indicadores gerais\n\n```{python}\n#| label: tbl-indicadores\n#| tbl-cap: "Indicadores gerais da população fictícia"\n\nindicadores = pd.DataFrame({\n    "Indicador": [\n        "Pacientes",\n        "Idade média",\n        "Hipertensão",\n        "Diabetes",\n        "Acompanhamento regular",\n        "Internação evitável",\n        "Consultas médias na APS",\n        "Custo médio anual",\n    ],\n    "Resultado": [\n        f"{len(df):,}".replace(",", "."),\n        f"{df[\'idade\'].mean():.1f} anos",\n        f"{100 * df[\'hipertensao\'].mean():.1f}%",\n        f"{100 * df[\'diabetes\'].mean():.1f}%",\n        f"{100 * df[\'acompanhamento_regular\'].mean():.1f}%",\n        f"{100 * df[\'internacao_evitavel_12m\'].mean():.1f}%",\n        f"{df[\'consultas_aps_12m\'].mean():.1f}",\n        f"R$ {df[\'custo_total_12m\'].mean():,.2f}"\n            .replace(",", "X").replace(".", ",").replace("X", "."),\n    ],\n})\n\ndisplay(indicadores)\n```\n\nA @tbl-indicadores resume os principais resultados para leitura gerencial.\n\n# Comparações entre grupos\n\n## Regiões\n\n```{python}\n#| label: tbl-regioes\n#| tbl-cap: "Indicadores segundo região fictícia"\n\npor_regiao = (\n    df.groupby("regiao", observed=True)\n    .agg(\n        Pacientes=("id_paciente", "count"),\n        Vulnerabilidade=("indice_vulnerabilidade", "mean"),\n        Acompanhamento=(\n            "acompanhamento_regular",\n            lambda x: 100 * x.mean(),\n        ),\n        Internação=(\n            "internacao_evitavel_12m",\n            lambda x: 100 * x.mean(),\n        ),\n        Custo=("custo_total_12m", "mean"),\n    )\n    .round(1)\n    .reset_index()\n    .rename(columns={\n        "regiao": "Região",\n        "Vulnerabilidade": "Vulnerab.",\n        "Acompanhamento": "Acomp. (%)",\n        "Internação": "Intern. (%)",\n        "Custo": "Custo médio",\n    })\n)\n\ndisplay(por_regiao)\n```\n\n```{python}\n#| label: fig-internacao-regiao\n#| fig-cap: "Percentual de pacientes com internação evitável segundo região fictícia."\n#| fig-width: 8\n#| fig-height: 4.8\n\ndados_figura = por_regiao.sort_values("Intern. (%)", ascending=False)\n\nfig, ax = plt.subplots()\nbarras = ax.bar(dados_figura["Região"], dados_figura["Intern. (%)"])\nax.set_ylabel("Internação evitável (%)")\nax.set_xlabel("")\nax.set_ylim(0, dados_figura["Intern. (%)"].max() * 1.22)\nax.bar_label(barras, fmt="%.1f%%", padding=3)\nplt.xticks(rotation=20, ha="right")\nplt.tight_layout()\nplt.show()\n```\n\nA @fig-internacao-regiao facilita a comparação visual e complementa a @tbl-regioes.\n\n## Acompanhamento regular\n\n```{python}\n#| label: fig-internacao-acompanhamento\n#| fig-cap: "Internação evitável segundo acompanhamento regular."\n#| fig-width: 7\n#| fig-height: 4.5\n\ncomparacao = (\n    df.dropna(subset=["acompanhamento_regular"])\n    .groupby("acompanhamento_regular", observed=True)\n    .agg(\n        Internação=(\n            "internacao_evitavel_12m",\n            lambda x: 100 * x.mean(),\n        )\n    )\n    .reset_index()\n)\n\ncomparacao["Grupo"] = comparacao["acompanhamento_regular"].map({\n    0: "Sem acompanhamento",\n    1: "Com acompanhamento",\n})\n\nfig, ax = plt.subplots(figsize=(7, 4.5))\nbarras = ax.bar(comparacao["Grupo"], comparacao["Internação"])\nax.set_ylabel("Internação evitável (%)")\nax.set_xlabel("")\nax.set_ylim(0, comparacao["Internação"].max() * 1.25)\nax.bar_label(barras, fmt="%.1f%%", padding=3)\nplt.tight_layout()\nplt.show()\n```\n\nA @fig-internacao-acompanhamento mostra uma frequência menor de internação no grupo com acompanhamento regular.\n\n# Fatores associados à internação\n\n## Método\n\nFoi ajustada uma regressão logística. O exercício é explicativo e não deve ser interpretado como análise causal.\n\n```{python}\n#| label: tbl-modelo\n#| tbl-cap: "Regressão logística para internação evitável"\n\ndf_modelo = (\n    df[\n        [\n            "internacao_evitavel_12m",\n            "idade",\n            "indice_vulnerabilidade",\n            "hipertensao",\n            "diabetes",\n            "acompanhamento_regular",\n            "consultas_urgencia_12m",\n        ]\n    ]\n    .dropna()\n)\n\nX = sm.add_constant(df_modelo.drop(columns="internacao_evitavel_12m"))\ny = df_modelo["internacao_evitavel_12m"]\nmodelo = sm.Logit(y, X).fit(disp=False)\n\nrotulos_modelo = {\n    "const": "Constante",\n    "idade": "Idade",\n    "indice_vulnerabilidade": "Vulnerabilidade",\n    "hipertensao": "Hipertensão",\n    "diabetes": "Diabetes",\n    "acompanhamento_regular": "Acompanhamento regular",\n    "consultas_urgencia_12m": "Consultas de urgência",\n}\n\nresultado = pd.DataFrame({\n    "Variável": [rotulos_modelo[v] for v in modelo.params.index],\n    "OR": np.exp(modelo.params.values),\n    "IC95% inf.": np.exp(modelo.conf_int()[0].values),\n    "IC95% sup.": np.exp(modelo.conf_int()[1].values),\n    "p": modelo.pvalues.values,\n}).round(3)\n\ndisplay(resultado)\n```\n\nA @tbl-modelo reúne estimativas, intervalos de confiança e valores de p em uma estrutura compacta.\n\n::: {.callout-note}\n## O que o Quarto acrescenta?\n\nO código continua sendo Python, mas o resultado ganha estrutura editorial, referências cruzadas e múltiplos formatos de entrega.\n:::\n\n# Conclusões\n\n1. A base fictícia simula problemas comuns da gestão em saúde.\n2. O Quarto integra texto, código, tabelas e gráficos.\n3. Os gráficos estáticos funcionam em HTML, Word, PDF e PowerPoint.\n4. Relatórios e apresentações devem compartilhar os cálculos, mas não necessariamente o mesmo layout.\n', encoding="utf-8")
arquivo_apresentacao.write_text('\n---\ntitle: "Indicadores da Atenção Primária"\nsubtitle: "Dados inteiramente fictícios"\nauthor: "Material didático"\ndate: today\nlang: pt-BR\nformat:\n  pptx:\n    slide-level: 2\n    incremental: false\nexecute:\n  echo: false\n  warning: false\n  error: false\njupyter: python3\n---\n\n```{python}\n#| label: setup\n#| include: false\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport statsmodels.api as sm\nfrom IPython.display import Markdown, display\n\ndf = pd.read_csv(Path("pacientes_ficticios.csv"))\n\nplt.rcParams.update({\n    "figure.figsize": (9, 5),\n    "figure.dpi": 140,\n    "axes.spines.top": False,\n    "axes.spines.right": False,\n})\n```\n\n## Sobre o exercício\n\n- Base com `{python} f"{len(df):,}".replace(",", ".")` pacientes fictícios\n- Contexto de atenção primária\n- Indicadores clínicos, assistenciais e financeiros\n- Objetivo: demonstrar análise reproduzível com Quarto\n\n## Indicadores gerais\n\n```{python}\n#| output: asis\n\nitens = [\n    f"**Hipertensão:** {100 * df[\'hipertensao\'].mean():.1f}%",\n    f"**Diabetes:** {100 * df[\'diabetes\'].mean():.1f}%",\n    f"**Acompanhamento regular:** {100 * df[\'acompanhamento_regular\'].mean():.1f}%",\n    f"**Internação evitável:** {100 * df[\'internacao_evitavel_12m\'].mean():.1f}%",\n    f"**Consultas médias na APS:** {df[\'consultas_aps_12m\'].mean():.1f}",\n]\n\nprint("\\n".join(f"- {item}" for item in itens))\n```\n\n## Internações evitáveis por região\n\n```{python}\n#| fig-width: 9\n#| fig-height: 5\n\npor_regiao = (\n    df.groupby("regiao", observed=True)["internacao_evitavel_12m"]\n    .mean()\n    .mul(100)\n    .sort_values(ascending=False)\n)\n\nfig, ax = plt.subplots(figsize=(9, 5))\nbarras = ax.bar(por_regiao.index, por_regiao.values)\nax.set_ylabel("Internação evitável (%)")\nax.set_xlabel("")\nax.set_ylim(0, por_regiao.max() * 1.22)\nax.bar_label(barras, fmt="%.1f%%", padding=3)\nplt.tight_layout()\nplt.show()\n```\n\n## Efeito do acompanhamento regular\n\n```{python}\n#| fig-width: 8.5\n#| fig-height: 5\n\ncomparacao = (\n    df.dropna(subset=["acompanhamento_regular"])\n    .groupby("acompanhamento_regular")["internacao_evitavel_12m"]\n    .mean()\n    .mul(100)\n)\n\nrotulos = ["Sem acompanhamento", "Com acompanhamento"]\n\nfig, ax = plt.subplots(figsize=(8.5, 5))\nbarras = ax.bar(rotulos, comparacao.values)\nax.set_ylabel("Internação evitável (%)")\nax.set_ylim(0, comparacao.max() * 1.25)\nax.bar_label(barras, fmt="%.1f%%", padding=3)\nplt.tight_layout()\nplt.show()\n```\n\n## Modelo explicativo\n\n```{python}\n#| output: asis\n\ndf_modelo = df[\n    [\n        "internacao_evitavel_12m",\n        "idade",\n        "indice_vulnerabilidade",\n        "hipertensao",\n        "diabetes",\n        "acompanhamento_regular",\n        "consultas_urgencia_12m",\n    ]\n].dropna()\n\nX = sm.add_constant(df_modelo.drop(columns="internacao_evitavel_12m"))\ny = df_modelo["internacao_evitavel_12m"]\nmodelo = sm.Logit(y, X).fit(disp=False)\n\nor_values = np.exp(modelo.params)\n\ndestaques = [\n    f"**Vulnerabilidade:** OR {or_values[\'indice_vulnerabilidade\']:.2f}",\n    f"**Hipertensão:** OR {or_values[\'hipertensao\']:.2f}",\n    f"**Diabetes:** OR {or_values[\'diabetes\']:.2f}",\n    f"**Acompanhamento regular:** OR {or_values[\'acompanhamento_regular\']:.2f}",\n    f"**Consultas de urgência:** OR {or_values[\'consultas_urgencia_12m\']:.2f}",\n]\n\nprint("\\n".join(f"- {item}" for item in destaques))\n```\n\n> Associação não implica causalidade. Os dados são simulados.\n\n## O que o Quarto permite\n\n- Uma fonte de dados e cálculos\n- Relatório HTML navegável\n- Documento Word editável\n- PDF técnico\n- Apresentação PowerPoint\n- Atualização automática após mudanças nos dados\n\n## Conclusão\n\nO mesmo processo analítico pode abastecer diferentes produtos.\n\nO conteúdo é compartilhado, mas o layout deve ser adaptado ao formato e ao público.\n', encoding="utf-8")

print(f"Relatório: {arquivo_relatorio}")
print(f"Apresentação: {arquivo_apresentacao}")


Relatório: /content/quarto_saude/relatorio_saude.qmd
Apresentação: /content/quarto_saude/apresentacao_saude.qmd


In [10]:
!quarto install tinytex --no-prompt

Installing tinytex
[✓] Downloading TinyTex v2026.08
[✓] Unzipping TinyTeX-linux-x86_64-v2026.08.tar.xz
[✓] Moving files
[✓] Verifying tlgpg support
[✓] Configuring font paths
[✓] Default Repository: https://tlnet.yihui.org
Installation successful


In [11]:

# ============================================================
# RENDERIZAÇÃO DOS FORMATOS
# ============================================================
# Para o PDF, instale o TinyTeX uma vez por sessão:

tarefas = [
    ("relatorio_saude.qmd", "html"),
    ("relatorio_saude.qmd", "docx"),
    ("relatorio_saude.qmd", "pdf"),
    ("apresentacao_saude.qmd", "pptx"),
]

resultados_renderizacao = []

for documento, formato in tarefas:
    comando = [
        "quarto",
        "render",
        documento,
        "--to",
        formato,
        "--execute",
    ]

    resultado = subprocess.run(
        comando,
        cwd=PASTA_PROJETO,
        text=True,
        capture_output=True,
    )

    sucesso = resultado.returncode == 0
    resultados_renderizacao.append(
        {
            "documento": documento,
            "formato": formato,
            "sucesso": sucesso,
            "mensagem": resultado.stderr[-1000:] if not sucesso else "OK",
        }
    )

    print("=" * 60)
    print(f"{documento} -> {formato.upper()}")

    if sucesso:
        nome_saida = (
            "apresentacao_saude.pptx"
            if formato == "pptx"
            else f"relatorio_saude.{formato}"
        )
        print(f"Gerado: {PASTA_PROJETO / nome_saida}")
    else:
        print(resultado.stderr)


relatorio_saude.qmd -> HTML
Gerado: /content/quarto_saude/relatorio_saude.html
relatorio_saude.qmd -> DOCX
Gerado: /content/quarto_saude/relatorio_saude.docx
relatorio_saude.qmd -> PDF
Gerado: /content/quarto_saude/relatorio_saude.pdf
apresentacao_saude.qmd -> PPTX
Gerado: /content/quarto_saude/apresentacao_saude.pptx


In [12]:

# Arquivos finais disponíveis para download no Colab.
from google.colab import files

arquivos_finais = [
    PASTA_PROJETO / "relatorio_saude.html",
    PASTA_PROJETO / "relatorio_saude.docx",
    PASTA_PROJETO / "relatorio_saude.pdf",
    PASTA_PROJETO / "apresentacao_saude.pptx",
]

for arquivo in arquivos_finais:
    print(
        f"{'OK' if arquivo.exists() else 'Não gerado'}: "
        f"{arquivo.name}"
    )

# Use uma linha por vez:
# files.download(str(PASTA_PROJETO / "relatorio_saude.html"))
# files.download(str(PASTA_PROJETO / "relatorio_saude.docx"))
# files.download(str(PASTA_PROJETO / "relatorio_saude.pdf"))
# files.download(str(PASTA_PROJETO / "apresentacao_saude.pptx"))


OK: relatorio_saude.html
OK: relatorio_saude.docx
OK: relatorio_saude.pdf
OK: apresentacao_saude.pptx



## **7. Por que esta versão funciona melhor?**


### Gráficos

O Plotly produz conteúdo interativo em HTML. Nesta versão, os gráficos do relatório e da apresentação são feitos com Matplotlib. O Quarto os converte em imagens estáticas compatíveis com HTML, Word, PDF e PowerPoint.

### PDF

O relatório usa margens definidas e tabelas com títulos mais curtos. O código fica oculto, evitando páginas ocupadas por blocos extensos.

### PowerPoint

A apresentação possui um arquivo próprio. Cada título de nível 2 inicia um slide, e os resultados são apresentados como gráficos ou listas curtas. Isso evita tentar encaixar um relatório completo dentro de slides.

### Princípio didático

O Quarto permite reutilizar dados e cálculos, mas não significa que todos os formatos devam ter exatamente a mesma estrutura visual.


## **8. Exercícios com respostas**


### Experimento A. Tornar o código visível no HTML

No YAML do relatório:

```yaml
execute:
  echo: true
```

Para manter o código recolhível:

```yaml
format:
  html:
    code-fold: true
```

### Experimento B. Alterar o tema do HTML

```yaml
format:
  html:
    theme: flatly
```

Outras opções incluem `cosmo`, `litera`, `minty` e `zephyr`.

### Experimento C. Adicionar um indicador

```python
df["consultas_aps_12m"].mean()
```

Esse valor pode ser inserido na tabela de indicadores ou no texto com código inline.

### Experimento D. Comparar grupos

A versão corrigida já inclui um gráfico de internação evitável entre pacientes com e sem acompanhamento regular.

### Experimento E. Gerar os formatos

```bash
quarto render relatorio_saude.qmd --to html --execute
quarto render relatorio_saude.qmd --to docx --execute
quarto render relatorio_saude.qmd --to pdf --execute
quarto render apresentacao_saude.qmd --to pptx --execute
```

### Pergunta para discussão

Por que não usar o mesmo layout para relatório e apresentação?

### Resposta

Porque os formatos cumprem funções diferentes. Relatórios permitem leitura detalhada, tabelas e metodologia. Apresentações precisam de síntese, hierarquia visual e uma mensagem principal por slide. O Quarto compartilha os cálculos, mas permite adaptar a comunicação.



## **9. Síntese**




O fluxo corrigido é:

```text
base fictícia
    ↓
cálculos em Python
    ├── relatório Quarto → HTML, Word e PDF
    └── apresentação Quarto → PowerPoint
```

Isso preserva a reprodutibilidade sem sacrificar a legibilidade de cada formato.